# Feature Analysis Notebook

Inspect engineered features, model feature importances, and SHAP interpretation artifacts produced by the pipeline.

In [ ]:
from pathlib import Path
import sys

import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_path, get_value, load_config

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
config = load_config(CONFIG_PATH)
PROCESSED_DIR = get_path(config, "processed_dir", base_dir=PROJECT_ROOT)
REPORTS_DIR = get_path(config, "reports_dir", base_dir=PROJECT_ROOT)
preview_rows = int(get_value(config, "eda", "preview_rows"))
print(f"Project root: {PROJECT_ROOT}")

## 1. Engineered Feature Frame

Review the saved feature frame and metadata summary.

In [ ]:
feature_frame_path = PROCESSED_DIR / str(get_value(config, "feature_engineering", "output_file"))
feature_metadata_path = PROCESSED_DIR / str(get_value(config, "feature_engineering", "metadata_file"))

print("Feature frame exists:", feature_frame_path.exists())
print("Feature metadata exists:", feature_metadata_path.exists())

if feature_frame_path.exists():
    feature_frame = pd.read_parquet(feature_frame_path)
    print("Feature frame shape:", feature_frame.shape)
    display(feature_frame.head(preview_rows))

if feature_metadata_path.exists():
    metadata = json.loads(feature_metadata_path.read_text(encoding="utf-8"))
    display({key: metadata[key] for key in ["row_count", "column_count"] if key in metadata})

## 2. Feature Importances

Review model-level feature importance outputs and the top-ranked XGBoost features.

In [ ]:
importance_path = REPORTS_DIR / "feature_importances.csv"
print("Feature importances exists:", importance_path.exists())

if importance_path.exists():
    feature_importances = pd.read_csv(importance_path)
    display(feature_importances.head(preview_rows))
    if "xgboost" in set(feature_importances["model"]):
        display(
            feature_importances[feature_importances["model"] == "xgboost"]
            .sort_values("importance_norm", ascending=False)
            .head(int(get_value(config, "eda", "top_categories")))
        )

## 3. SHAP Summaries

Inspect SHAP ranking and available plots for both global importance and directionality.

In [ ]:
shap_artifacts = {
    "summary": REPORTS_DIR / "shap_summary.csv",
    "summary_sample": REPORTS_DIR / "shap_summary_sample.csv",
    "top20": REPORTS_DIR / "shap_top20.png",
    "top20_sample": REPORTS_DIR / "shap_top20_sample.png",
    "beeswarm": REPORTS_DIR / "shap_beeswarm.png",
    "beeswarm_sample": REPORTS_DIR / "shap_beeswarm_sample.png",
}

for name, path in shap_artifacts.items():
    print(f"{name}: {path.exists()} - {path}")

for table_key in ["summary", "summary_sample"]:
    path = shap_artifacts[table_key]
    if path.exists():
        print(f"
{table_key}")
        display(pd.read_csv(path).head(int(get_value(config, "eda", "top_categories"))))

for image_key in ["top20", "top20_sample", "beeswarm", "beeswarm_sample"]:
    path = shap_artifacts[image_key]
    if path.exists():
        display(Image(filename=str(path)))

## 4. Artifact Summary

A compact checklist of interpretation artifacts expected after running the analysis scripts.

In [ ]:
artifacts = [
    REPORTS_DIR / "feature_importances.csv",
    REPORTS_DIR / "shap_summary.csv",
    REPORTS_DIR / "shap_summary_sample.csv",
    REPORTS_DIR / "shap_top20.png",
    REPORTS_DIR / "shap_top20_sample.png",
    REPORTS_DIR / "shap_beeswarm.png",
    REPORTS_DIR / "shap_beeswarm_sample.png",
]

artifact_status = pd.DataFrame({"artifact": [str(path) for path in artifacts], "exists": [path.exists() for path in artifacts]})
display(artifact_status)